# EMG2MCU package example

This notebook creates a deterministic synthetic three-channel EMG recording at runtime and runs the package-backed AutoML example on it. No real participant data is needed or included. The synthetic signals are for software demonstration only, not physiological analysis.

Run this notebook from the repository root after installing the project with `python -m pip install -e .`.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np

import EMG2MCU

In [ ]:
fs = 1000
rng = np.random.default_rng(42)
signal_blocks = []
label_blocks = []

def add_segment(label, duration_s, amplitudes):
    sample_count = int(duration_s * fs)
    time = np.arange(sample_count) / fs
    segment = rng.normal(0, 0.015, size=(3, sample_count))
    if label >= 0:
        fade = np.minimum(1, np.minimum(time / 0.15, (duration_s - time) / 0.15))
        frequencies = np.array([83, 117, 151]) + 7 * label
        carriers = np.sin(2 * np.pi * frequencies[:, None] * time)
        carriers += 0.35 * np.sin(2 * np.pi * frequencies[:, None] * 1.7 * time)
        segment += np.asarray(amplitudes)[:, None] * fade * carriers
    signal_blocks.append(segment.T)
    label_blocks.append(np.full(sample_count, label, dtype=int))

amplitude_patterns = [
    [0.8, 0.15, 0.15],
    [0.15, 0.8, 0.15],
    [0.15, 0.15, 0.8],
]

add_segment(-1, 2.0, [0, 0, 0])
for _ in range(4):
    for label, amplitudes in enumerate(amplitude_patterns):
        add_segment(label, 3.0, amplitudes)
        add_segment(-1, 0.5, [0, 0, 0])

labels = np.concatenate(label_blocks)
signals = np.concatenate(signal_blocks, axis=0)
recording = np.column_stack(
    [np.arange(len(labels)), np.zeros((len(labels), 2)), labels,
     np.zeros(len(labels)), signals]
)

synthetic_data_dir = tempfile.TemporaryDirectory(prefix="emg2mcu-example-")
synthetic_data_path = Path(synthetic_data_dir.name) / "synthetic_emg.txt"
with synthetic_data_path.open("w", encoding="utf-8") as recording_file:
    recording_file.write("Synthetic EMG recording generated by the example\nEndOfHeader\n")
    np.savetxt(recording_file, recording, delimiter="\t", fmt="%.6f")

data_cfg = EMG2MCU.DataConfig(
    data_file=synthetic_data_path.name,
    data_path=str(synthetic_data_path.parent),
    participant="SyntheticDemo",
    label_names=[
        "Peito",
        "Trapézio",
        "Pescoço",
    ],
    channel_names=["Peito", "Trapézio", "Pescoço"],
)

preprocessing_cfg = EMG2MCU.PreprocessingConfig(
    fs=fs,
    lowcut=30,
    highcut=450,
    notch_freqs=(60,),
    Q=30,
    signal_padding=0.2,
    butter_order=4,
    adc_pin=34,
)

events_cfg = EMG2MCU.EventsConfig(
    smooth_win_size=500,
    onset_sustain_ms=1500,
    onset_tol_ratio=0.20,
    pre_onset_removal_ratio=0.25,
    event_pad_ms=0,
)

windowing_cfg = EMG2MCU.WindowingConfig(
    test_size=0.25,
    window_ms=50,
    overlap=0.0,
)

feature_cfg = EMG2MCU.FeatureConfig(
    features=["RMS"],
    return_dataframe=True,
)

normalize_cfg = EMG2MCU.NormalizeConfig(scale_y=False)

classify_cfg = EMG2MCU.ClassificationConfig(
    algorithms=[
        "lightgbm",
        "xgboost",
        "catboost",
        "random_forest",
        "extra_trees",
        "svm",
        "mlp",
        "logistic_regression",
    ],
    sort_by="accuracy",
)

In [ ]:
try:
    results = EMG2MCU.automl(
        data_cfg=data_cfg,
        preprocessing_cfg=preprocessing_cfg,
        events_cfg=events_cfg,
        windowing_cfg=windowing_cfg,
        feature_cfg=feature_cfg,
        normalize_cfg=normalize_cfg,
        classify_cfg=classify_cfg,
        run_preprocessing=True,
        run_windowing=True,
        run_events=True,
        run_plots=True,
        run_features=True,
        run_normalization=True,
        run_classification=True,
        output_dir=Path("output_emg2mcu"),
    )
finally:
    synthetic_data_dir.cleanup()